# Factor Investing: Alpha Concentration versus Diversification

**Authors:** Lars Heinrich, Antoniya Shivarova, Martin Žurek

**Published:** 2021-06-02

**URL:** [https://doi.org/10.1057/s41260-021-00226-0](https://doi.org/10.1057/s41260-021-00226-0)

**Abstract:** Despite extensive research support, the role of diversification in current factor investing strategies remains neglected. This paper investigates whether well-designed multifactor portfolios should not only be based on firm characteristics, but should also include portfolio diversification effects. While the alpha concentration approach mainly considers factor-specific firm characteristics, the diversified approach utilizes covariance estimators in addition to firm characteristics to account for portfolio diversification. The corresponding out-of-sample results show that including an efficient covariance estimator improves the performance of long-only multifactor portfolios compared to the pure alpha concentration approach. A particular advantage of diversified factor investing strategies can be identified in the significant increase in exposure to the low-volatility factor represented by firm characteristics with high informational content. No significant performance differences are observed for long-short portfolios where the factor exposures of the alpha concentration and diversification approaches are similar with respect to the low-volatility factor.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration parameters
start_date = '2015-01-01'
end_date = '2023-01-01'
ticker = 'AAPL'
lookback_period = 252  # 1 year of trading days
risk_free_rate = 0.01  # 1% annual risk-free rate

## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download data
data = yf.download(ticker, start=start_date, end=end_date)

# Feature engineering
data['Returns'] = data['Adj Close'].pct_change()
data['Volatility'] = data['Returns'].rolling(window=lookback_period).std() * np.sqrt(252)

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Signal generation: simple moving average crossover
short_window = 40
long_window = 100

data['Short_MA'] = data['Adj Close'].rolling(window=short_window, min_periods=1).mean()
data['Long_MA'] = data['Adj Close'].rolling(window=long_window, min_periods=1).mean()
data['Signal'] = 0

data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1

data['Position'] = data['Signal'].shift(1)  # Shift signals to avoid look-ahead bias

## Phase 4: Vectorized Backtest

In [ ]:
# Calculate strategy returns
data['Strategy_Returns'] = data['Position'] * data['Returns']

# Calculate cumulative returns
data['Cumulative_Strategy_Returns'] = (1 + data['Strategy_Returns']).cumprod()
data['Cumulative_Market_Returns'] = (1 + data['Returns']).cumprod()

## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
sharpe_ratio = (data['Strategy_Returns'].mean() - risk_free_rate / 252) / data['Strategy_Returns'].std() * np.sqrt(252)
sortino_ratio = (data['Strategy_Returns'].mean() - risk_free_rate / 252) / data[data['Strategy_Returns'] < 0]['Strategy_Returns'].std() * np.sqrt(252)
max_drawdown = (data['Cumulative_Strategy_Returns'].cummax() - data['Cumulative_Strategy_Returns']).max()
calmar_ratio = (data['Strategy_Returns'].mean() * 252) / max_drawdown

# Plot equity curve
plt.figure(figsize=(12, 6))
plt.plot(data['Cumulative_Strategy_Returns'], label='Strategy')
plt.plot(data['Cumulative_Market_Returns'], label='Market')
plt.title('Equity Curve')
plt.legend()
plt.show()

# Print performance metrics
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')

## Phase 6: Monitoring Stub

In [ ]:
# Monitoring stub - to be implemented
# This would typically involve setting up alerts or dashboards to monitor the strategy in real-time
pass